# 1. 설치 검증

In [1]:
import rustima
import numpy as np

print(rustima.version())                              # "0.1.0"
y = np.random.randn(100).cumsum()
r = rustima.sarimax_fit(y, order=(1, 1, 1), seasonal=(0, 0, 0, 0))
print(f"converged={r['converged']}, AIC={r['aic']:.2f}")

0.1.0
converged=True, AIC=303.92


# 2. 예제

## 2.1. 첫 예측(5분 워크스루)
- converged=True → 최적화 성공
- 낮은 AIC / BIC → 더 나은 모델 적합 (다른 차수 대비 상대적으로)
- Ljung-Box p > 0.05 → 잔차가 백색잡음처럼 보임 (good)
- ci_lower / ci_upper → 불확실성 밴드; 넓을수록 덜 확신

In [8]:
import numpy as np
from rustima import SARIMAXModel, auto_arima

# ── 1. 트렌드 + 1년 계절성이 있는 월별 매출 데이터 시뮬레이션 ────────────
rng = np.random.default_rng(42)
n = 120  # 10년치 월별
trend = 0.5 * np.arange(n)                           # 선형 상승 트렌드
season = 10 * np.sin(2 * np.pi * np.arange(n) / 12)  # 1년 주기 (s=12)
noise = rng.normal(0, 1.0, n)
y = trend + season + noise

# ── 2. auto_arima가 차수를 알아서 선택 ────────────────────────────────────
auto_result = auto_arima(y, s=12, trace=True)  # trace=True → 시도한 모델 출력
print(auto_result.search_summary())
print("\n")
# >>> Best: SARIMA(0,1,1)(0,1,1)[12]  AIC=345.67  (evaluated 23 models)

# ── 3. 선택된 모델 살펴보기 ────────────────────────────────────────────────
model = auto_result.result              # SARIMAXResult 객체
print(model.summary())                  # statsmodels 스타일 파라미터 테이블
print(f"AIC={model.aic:.2f}  BIC={model.bic:.2f}")
print("\n")

# ── 4. 다음 12개월 95% 신뢰구간으로 예측 ──────────────────────────────────
forecast = model.forecast(steps=12, alpha=0.05)
df = forecast.to_dataframe()            # Polars DataFrame
print(df)
# 형태 (12, 5): step | mean | variance | ci_lower | ci_upper

# ── 5. 잔차 검사 (랜덤 노이즈처럼 보여야 함) ──────────────────────────────
diag = model.diagnostics()
print(f"Ljung-Box p-value: {diag['ljung_box_pvalue']:.3f}  (>0.05 이면 good)")

  ARIMA(0,1,0)(0,1,0)[12] : aic=395.332
  ARIMA(2,1,2)(0,1,0)[12] : aic=328.028
  ARIMA(1,1,0)(0,1,0)[12] : aic=361.325
  ARIMA(0,1,1)(0,1,0)[12] : aic=322.248
  ARIMA(0,1,0)(1,1,0)[12] : aic=376.731
  ARIMA(0,1,0)(0,1,1)[12] : aic=369.145
  ARIMA(1,1,0)(1,1,0)[12] : aic=337.221
  ARIMA(0,1,1)(0,1,1)[12] : aic=294.530
  ARIMA(1,1,1)(0,1,1)[12] : aic=296.545
  ARIMA(0,1,2)(0,1,1)[12] : aic=296.574
  ARIMA(0,1,1)(1,1,1)[12] : aic=294.959
  ARIMA(0,1,1)(0,1,2)[12] : aic=295.058
  ARIMA(1,1,2)(0,1,1)[12] : aic=297.474
auto_arima: Best ARIMA(0,1,1)(0,1,1)[12]
  aic=294.530
  Models evaluated: 13 (13 converged)


                               SARIMAX Results                                
Model: SARIMAX(0,1,1)(0,1,1)[12]                  Log Likelihood:     -144.265
No. Observations: 120                              AIC:                294.530
Trend: n                                           BIC:                302.892
Method: lbfgsb                                     HQIC:             

## 2.2 저수준 API

In [ ]:
import numpy as np
import polars as pl
import rustima

y = np.random.randn(200).cumsum()

# 1. 모델 적합
result = rustima.sarimax_fit(y, order=(1, 1, 1), seasonal=(0, 0, 0, 0))
print(f"Converged: {result['converged']}, AIC: {result['aic']:.2f}")

# 2. 10스텝 앞 예측
fc = rustima.sarimax_forecast(
    y, order=(1, 1, 1), seasonal=(0, 0, 0, 0),
    params=np.array(result["params"]), steps=10
)
print(f"Forecast: {fc['mean'][:5]}")

# 3. 잔차 진단 (DataFrame 표 형식)
res = rustima.sarimax_residuals(
    y, order=(1, 1, 1), seasonal=(0, 0, 0, 0),
    params=np.array(result["params"])
)

res_df = pl.DataFrame({
    "index": np.arange(len(res["residuals"])),
    "residuals": res["residuals"],
    "standardized_residuals": res["standardized_residuals"],
})
print(res_df)

Converged: True, AIC: 556.83
Forecast: [13.224001034760711, 13.270774834518685, 13.256322704626896, 13.260788111745622, 13.259408393954178]
shape: (200, 3)
┌───────┬───────────┬────────────────────────┐
│ index ┆ residuals ┆ standardized_residuals │
│ ---   ┆ ---       ┆ ---                    │
│ i64   ┆ f64       ┆ f64                    │
╞═══════╪═══════════╪════════════════════════╡
│ 0     ┆ -0.058473 ┆ -0.000043              │
│ 1     ┆ -0.362149 ┆ -0.000365              │
│ 2     ┆ 0.763519  ┆ 0.775623               │
│ 3     ┆ 0.03876   ┆ 0.040004               │
│ 4     ┆ 0.186927  ┆ 0.193027               │
│ …     ┆ …         ┆ …                      │
│ 195   ┆ -0.928526 ┆ -0.958843              │
│ 196   ┆ 0.994839  ┆ 1.027322               │
│ 197   ┆ 1.391526  ┆ 1.43696                │
│ 198   ┆ 0.327482  ┆ 0.338174               │
│ 199   ┆ 1.205803  ┆ 1.245174               │
└───────┴───────────┴────────────────────────┘


## 2.3. 고수준 API(SARIMAXModel - statsmodels 호환)

In [31]:
import numpy as np
import polars as pl
from rustima import SARIMAXModel

model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="c")
result = model.fit()

# 파라미터 테이블 요약 (빠름, 추론 통계 없음)
print(result.summary())
print("\n")

# Hessian 기반 추론 포함 요약 (std err, z, p-value, CI)
print(result.summary(inference="hessian"))
print("\n")

# Hessian vs statsmodels 추론을 나란히 비교
print(result.summary(inference="both"))
print("\n")

# Polars DataFrame으로 파라미터 테이블
pt = result.params_table(inference="hessian")
print("── Parameter DataFrame ──")
print(pt)  # shape: (k, 7) — name, coef, std_err, z, p_value, ci_lower, ci_upper
print(f"AIC: {result.aic:.2f}, BIC: {result.bic:.2f}, HQIC: {result.hqic:.2f}")
print("\n")

# 신뢰구간 포함 예측 + Polars DataFrame (alpha=0.05 + alpha=0.10 CI 병합)
fcast = result.forecast(steps=10, alpha=0.05)
df = fcast.to_dataframe()  # Polars: step, mean, variance, ci_lower, ci_upper
ci = fcast.conf_int()          # (10, 2) 배열 [lower, upper] — 출력 생략 (df와 중복)
ci_90 = fcast.conf_int(0.10)   # 다른 alpha로 재계산
df = df.with_columns([
    pl.Series("ci_90_lower", ci_90[:, 0]),
    pl.Series("ci_90_upper", ci_90[:, 1]),
])
print("── Forecast DataFrame (alpha=0.05 + ci_90) ──")
print(df)
print("\n")

# In-sample 예측 + 표준화 잔차
pred = result.get_prediction(start=0, end=210)
pred_df = pred.to_dataframe()  # Polars: index, predicted_mean
residuals = result.resid
resid_df = pl.DataFrame({
    "index": np.arange(len(residuals)),
    "resid": residuals,
})
pred_resid_df = pred_df.join(resid_df, on="index", how="left").with_columns(
    pl.when(pl.col("resid").is_null())
      .then(pl.lit("-"))
      .otherwise(pl.col("resid").cast(pl.Utf8))
      .alias("resid")
)
print("── In-sample Prediction + Standardized Residuals ──")
print(pred_resid_df)
print("\n")

# 잔차 진단 (Ljung-Box, Jarque-Bera, 이분산) — (2, N) 형태: metric row + value row
diag = result.diagnostics()
diag_flat = {}
for k, v in diag.items():
    if isinstance(v, (list, tuple, np.ndarray)):
        for i, vv in enumerate(np.atleast_1d(v)):
            diag_flat[f"{k}[{i}]"] = float(vv)
    else:
        diag_flat[k] = float(v)

diag_df = pl.DataFrame({
    "value": list(diag_flat.values())
}).transpose(include_header=False)
diag_df.columns = list(diag_flat.keys())
print("── Diagnostics ──")
print(diag_df)

                               SARIMAX Results                                
Model: SARIMAX(1,1,1)(0,0,0)[0]                   Log Likelihood:     -274.832
No. Observations: 200                              AIC:                557.664
Trend: c                                           BIC:                570.857
Method: lbfgsb                                     HQIC:               563.003
Converged: True                                     Scale:            0.926962
Date: 2026-04-20                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
       intercept     0.0832
           ar.L1    -0.2569
           ma.L1     0.1235


                               SARIMAX Results                                
Model: SARIMAX(1,1,1)(0,0,0)[0]                   Log Likelihood:     -274.832
No. Observations:

## 2.4. auto_arima - 자동 차수 선택

In [23]:
import warnings
from rustima import auto_arima

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)

    # Stepwise (Hyndman-Khandakar, 기본)
    res = auto_arima(y, max_p=5, max_q=5, s=12, stepwise=True, trace=True)
    print(res.summary())           # statsmodels 스타일 전체 요약 + 추론 통계
    print("\n")
    print(res.search_summary())    # 짧은 3줄 요약 (차수, IC, 모델 수)
    print("\n")
    print(res.result.forecast(steps=12).to_dataframe())
    print("\n")

    # Grid Search (Rayon 병렬 — 차수 조합별 fit job 분산)
    res = auto_arima(y, max_p=3, max_q=3, s=7, stepwise=False, criterion="bic")
    print(res.summary())
    print("\n")

    # 탐색 이력 (Polars DataFrame)
    print(res.history_dataframe())

  ARIMA(0,1,0)(0,1,0)[12] : aic=650.312
  ARIMA(2,1,2)(0,1,0)[12] : aic=651.006
  ARIMA(1,1,0)(0,1,0)[12] : aic=645.394
  ARIMA(0,1,1)(0,1,0)[12] : aic=645.335
  ARIMA(0,1,0)(1,1,0)[12] : aic=590.039
  ARIMA(0,1,0)(0,1,1)[12] : aic=579.890
  ARIMA(1,1,0)(1,1,0)[12] : aic=584.781
  ARIMA(0,1,1)(0,1,1)[12] : aic=553.163
  ARIMA(1,1,1)(0,1,1)[12] : aic=555.359
  ARIMA(0,1,2)(0,1,1)[12] : aic=555.387
  ARIMA(0,1,1)(1,1,1)[12] : aic=554.607
  ARIMA(0,1,1)(0,1,2)[12] : aic=554.580
  ARIMA(1,1,2)(0,1,1)[12] : aic=557.318
                               SARIMAX Results                                
Model: SARIMAX(0,1,1)(0,1,1)[12]                  Log Likelihood:     -273.582
No. Observations: 200                              AIC:                553.163
Trend: n                                           BIC:                563.058
Method: lbfgsb                                     HQIC:               557.167
Converged: True                                     Scale:            0.993010
Date: 

## 2.5. Trend(추세) 지원

In [24]:
# 상수항 (intercept)
model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="c")
# 선형 추세 (drift)
model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="t")
# 상수 + 선형 (intercept + drift)
model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="ct")

result = model.fit()
print(result.param_names)  # ['intercept', 'drift', 'ar.L1', 'ma.L1'] (trend='ct')

['intercept', 'drift', 'ar.L1', 'ma.L1']


## 2.6. 외생 회귀변수 사용

In [ ]:
import numpy as np

# 외생 회귀변수(2개) 생성
X_train = np.column_stack([np.arange(200), np.random.randn(200)])  # (200, 2)
X_future = np.column_stack([np.arange(200, 210), np.random.randn(10)])  # (10, 2)

model = SARIMAXModel(y, order=(1, 0, 1), seasonal_order=(0, 0, 0, 0), exog=X_train)
result = model.fit()
fcast = result.forecast(steps=10, exog=X_future)

## 2.7. 배치 병렬 처리

In [32]:
# 100개 시계열을 job 단위로 병렬 적합 (Rayon 멀티스레드)
series_list = [np.random.randn(200) for _ in range(100)]

results = rustima.sarimax_batch_fit(
    series_list, order=(1, 0, 0), seasonal=(0, 0, 0, 0)
)

for i, r in enumerate(results):
    print(f"Series {i}: converged={r['converged']}, AIC={r['aic']:.2f}")

# 시계열별 파라미터로 배치 예측
params_list = [np.array(r["params"]) for r in results]
forecasts = rustima.sarimax_batch_forecast(
    series_list, order=(1, 0, 0), seasonal=(0, 0, 0, 0),
    params_list=params_list, steps=10, alpha=0.05,
)

Series 0: converged=True, AIC=590.17
Series 1: converged=True, AIC=592.22
Series 2: converged=True, AIC=616.25
Series 3: converged=True, AIC=558.50
Series 4: converged=True, AIC=575.57
Series 5: converged=True, AIC=565.28
Series 6: converged=True, AIC=584.29
Series 7: converged=True, AIC=597.96
Series 8: converged=True, AIC=578.38
Series 9: converged=True, AIC=582.28
Series 10: converged=True, AIC=580.94
Series 11: converged=True, AIC=574.34
Series 12: converged=True, AIC=575.19
Series 13: converged=True, AIC=535.51
Series 14: converged=True, AIC=572.70
Series 15: converged=True, AIC=577.36
Series 16: converged=True, AIC=588.23
Series 17: converged=True, AIC=523.27
Series 18: converged=True, AIC=585.07
Series 19: converged=True, AIC=520.74
Series 20: converged=True, AIC=559.20
Series 21: converged=True, AIC=586.86
Series 22: converged=True, AIC=574.88
Series 23: converged=True, AIC=564.15
Series 24: converged=True, AIC=550.90
Series 25: converged=True, AIC=515.23
Series 26: converged=T

## 2.8. Grid Search 병렬 처리

In [33]:
# 여러 ARIMA 차수를 Rayon으로 한꺼번에 적합
results = rustima.sarimax_grid_search(
    y,
    order_list=[(0,1,0), (1,1,0), (1,1,1), (2,1,1)],
    seasonal_list=[(0,0,0,0)] * 4,
    trend="c",
)
for r in results:
    if "error" not in r:
        print(f"{r['order']}: AIC={r['aic']:.3f}")

(0, 1, 0): AIC=557.337
(1, 1, 0): AIC=555.710
(1, 1, 1): AIC=557.664
(2, 1, 1): AIC=559.687
